# Pitcher appearances

Inspect and validate the reusable pitcher-appearance view derived from local Statcast pitch records. The view has one row per pitcher, pitching team, and game; starts and relief appearances are classified at the appearance level.

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

from crooked_numbers_analytics.appearances import (
    create_pitcher_appearances_view,
)
from crooked_numbers_analytics.db import open_database

con = open_database()

## 2. Research parameters

Set `seasons = None` to use every locally available season. Change `team` for manual inspection without rebuilding the view.

In [2]:
seasons = [2023, 2024]
team = "MIL"

## 3. Create and inspect the view

The helper reads the selected Hive partitions directly and defaults to regular-season games (`game_type = 'R'`). It does not materialize another dataset.

In [3]:
create_pitcher_appearances_view(con, seasons=seasons)

In [4]:
con.execute(
    """
    SELECT *
    FROM pitcher_appearances
    WHERE pitching_team = ?
    ORDER BY game_date, game_pk, is_start DESC, first_inning, pitcher_id
    LIMIT 100
    """,
    [team],
).df()

,game_pk,game_date,season,pitching_team,pitcher_id,pitcher_name,is_start,pitch_count,first_inning,last_inning
0,718777,2023-03-30,2023,MIL,669203,"Burnes, Corbin",True,87,1,5
1,718777,2023-03-30,2023,MIL,657265,"Strzelecki, Peter",False,10,6,6
2,718777,2023-03-30,2023,MIL,681402,"Varland, Gus",False,19,7,7
3,718777,2023-03-30,2023,MIL,642770,"Guerra, Javy",False,8,8,8
4,718758,2023-04-01,2023,MIL,605540,"Woodruff, Brandon",True,97,1,6
...,...,...,...,...,...,...,...,...,...,...
95,718428,2023-04-25,2023,MIL,665625,"Peguero, Elvis",False,34,4,6
96,718428,2023-04-25,2023,MIL,606303,"Payamps, Joel",False,31,7,8
97,718428,2023-04-25,2023,MIL,657265,"Strzelecki, Peter",False,10,9,9
98,718419,2023-04-26,2023,MIL,642547,"Peralta, Freddy",True,98,1,6


## 4. Coverage and sanity checks

`pitch_count` is the number of Statcast pitch records in an appearance. Compare edge cases with official pitch counts before treating the values as perfectly equivalent.

In [5]:
con.execute(
    """
    SELECT
        season,
        COUNT(DISTINCT game_pk) AS games,
        COUNT(*) AS appearances,
        COUNT(*) FILTER (WHERE is_start) AS starts,
        COUNT(*) FILTER (WHERE NOT is_start) AS relief_appearances,
        MIN(pitch_count) AS min_pitch_count,
        MAX(pitch_count) AS max_pitch_count
    FROM pitcher_appearances
    WHERE pitching_team = ?
    GROUP BY season
    ORDER BY season
    """,
    [team],
).df()

,season,games,appearances,starts,relief_appearances,min_pitch_count,max_pitch_count
0,2023,162,680,162,518,1,110
1,2024,162,681,162,519,3,111


In [6]:
con.sql(
    """
    SELECT game_pk, pitching_team, pitcher_id, COUNT(*) AS rows
    FROM pitcher_appearances
    GROUP BY game_pk, pitching_team, pitcher_id
    HAVING COUNT(*) > 1
    ORDER BY game_pk, pitching_team, pitcher_id
    """
).df()

,game_pk,pitching_team,pitcher_id,rows


In [7]:
con.sql(
    """
    WITH starters AS (
        SELECT
            game_pk, pitching_team,
            COUNT(*) FILTER (WHERE is_start) AS starter_count
        FROM pitcher_appearances
        GROUP BY game_pk, pitching_team
    )
    SELECT *
    FROM starters
    WHERE starter_count <> 1
    ORDER BY game_pk, pitching_team
    """
).df()

,game_pk,pitching_team,starter_count


In [8]:
con.sql(
    """
    SELECT *
    FROM pitcher_appearances
    WHERE pitcher_id IS NULL
       OR pitching_team IS NULL
       OR pitch_count = 0
    ORDER BY game_date, game_pk
    """
).df()

,game_pk,game_date,season,pitching_team,pitcher_id,pitcher_name,is_start,pitch_count,first_inning,last_inning


## 5. Manual game inspection

Choose a game from the team sample and inspect both clubs, with starts first and inning spans visible.

In [9]:
inspection_game_pk = con.execute(
    """
    SELECT game_pk
    FROM pitcher_appearances
    WHERE pitching_team = ?
    ORDER BY game_date, game_pk
    LIMIT 1
    """,
    [team],
).fetchone()[0]

con.execute(
    """
    SELECT *
    FROM pitcher_appearances
    WHERE game_pk = ?
    ORDER BY pitching_team, is_start DESC, first_inning, pitcher_id
    """,
    [inspection_game_pk],
).df()

,game_pk,game_date,season,pitching_team,pitcher_id,pitcher_name,is_start,pitch_count,first_inning,last_inning
0,718777,2023-03-30,2023,CHC,573186,"Stroman, Marcus",True,91,1,6
1,718777,2023-03-30,2023,CHC,624522,"Thompson, Keegan",False,14,7,7
2,718777,2023-03-30,2023,CHC,502202,"Boxberger, Brad",False,14,8,8
3,718777,2023-03-30,2023,CHC,605242,"Fulmer, Michael",False,15,9,9
4,718777,2023-03-30,2023,MIL,669203,"Burnes, Corbin",True,87,1,5
5,718777,2023-03-30,2023,MIL,657265,"Strzelecki, Peter",False,10,6,6
6,718777,2023-03-30,2023,MIL,681402,"Varland, Gus",False,19,7,7
7,718777,2023-03-30,2023,MIL,642770,"Guerra, Javy",False,8,8,8
